# 🎬 AI Video Dubbing - Automatico desde Google Sheets

## Setup (solo la primera vez):
1. Crea una Google Sheet con el nombre **"Doblajes"**
2. En la columna A pon: `URL` | En la columna B pon: `Estado` | En la columna C pon: `Link Drive`
3. Pega un link de YouTube en A2
4. Dale **Runtime > Run all** a este Colab
5. El video doblado aparece en tu Google Drive en la carpeta **"VideosTraducidos"**

---

In [ ]:
#@title 1. Instalar dependencias (tarda ~2 min la primera vez)
!pip install -q yt-dlp openai-whisper edge-tts moviepy gspread google-auth
!apt-get -qq install -y ffmpeg > /dev/null 2>&1
print("✅ Dependencias instaladas")

In [ ]:
#@title 2. Conectar con Google Drive y Sheets
from google.colab import auth, drive
import gspread
from google.auth import default

# Autenticar (te pide permiso una sola vez)
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Montar Drive
drive.mount('/content/drive')

# Crear carpeta de salida en Drive
import os
DRIVE_OUTPUT = '/content/drive/MyDrive/VideosTraducidos'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

print("✅ Google Drive y Sheets conectados")
print(f"📁 Los videos se guardan en: Google Drive > VideosTraducidos")

In [ ]:
#@title 3. Leer URLs de Google Sheets
#@markdown Nombre de tu Google Sheet:
SHEET_NAME = "Doblajes" #@param {type:"string"}

try:
    sheet = gc.open(SHEET_NAME).sheet1
    all_rows = sheet.get_all_values()
    
    # Encontrar URLs pendientes (columna A con URL, columna B vacia o != "Listo")
    pending = []
    for i, row in enumerate(all_rows[1:], start=2):  # Skip header
        url = row[0].strip() if len(row) > 0 else ""
        status = row[1].strip() if len(row) > 1 else ""
        if url and status != "Listo" and ("youtube" in url or "youtu.be" in url):
            pending.append({"row": i, "url": url})
    
    print(f"📋 Sheet '{SHEET_NAME}' encontrada")
    print(f"🎬 {len(pending)} video(s) pendiente(s) por doblar:")
    for p in pending:
        print(f"   Fila {p['row']}: {p['url']}")
        
except gspread.SpreadsheetNotFound:
    print(f"❌ No se encontro la Sheet '{SHEET_NAME}'")
    print(f"   Crea una Google Sheet con ese nombre y pon URLs de YouTube en la columna A")

In [ ]:
#@title 4. Funciones del pipeline de doblaje
import subprocess
import json
import asyncio
import edge_tts
import whisper
import shutil
from pathlib import Path

TEMP_DIR = '/content/temp_dubbing'
WHISPER_MODEL = None

def download_video(url, output_dir):
    """Descarga video de YouTube."""
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, 'video.mp4')
    cmd = ['yt-dlp', '-f', 'bestvideo[height<=720]+bestaudio/best[height<=720]',
           '--merge-output-format', 'mp4', '-o', output_path, '--no-playlist', url]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        cmd = ['yt-dlp', '-f', 'best[height<=720]', '-o', output_path, '--no-playlist', url]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(f"Error descargando: {result.stderr}")
    return output_path

def extract_audio(video_path):
    """Extrae audio WAV del video."""
    audio_path = video_path.replace('.mp4', '.wav')
    cmd = ['ffmpeg', '-y', '-i', video_path, '-vn', '-acodec', 'pcm_s16le',
           '-ar', '16000', '-ac', '1', audio_path]
    subprocess.run(cmd, capture_output=True, text=True, check=True)
    return audio_path

def transcribe_audio(audio_path):
    """Transcribe con Whisper."""
    global WHISPER_MODEL
    if WHISPER_MODEL is None:
        print("   Cargando modelo Whisper (solo la primera vez)...")
        WHISPER_MODEL = whisper.load_model('base')
    result = WHISPER_MODEL.transcribe(audio_path, language=None)
    segments = []
    for i, seg in enumerate(result['segments']):
        segments.append({
            'id': i,
            'start': seg['start'],
            'end': seg['end'],
            'text': seg['text'].strip()
        })
    return segments, result.get('language', 'unknown')

def rewrite_segments_spanish(segments):
    """Reescribe segmentos manteniendo el texto (ya esta en español o se mantiene original)."""
    for seg in segments:
        seg['new_text'] = seg['text']
    return segments

async def generate_tts_segment(text, output_path, voice='es-MX-JorgeNeural', rate='+0%'):
    """Genera audio TTS para un segmento."""
    communicate = edge_tts.Communicate(text, voice, rate=rate)
    await communicate.save(output_path)

async def generate_all_tts(segments, audio_dir, voice='es-MX-JorgeNeural'):
    """Genera TTS para todos los segmentos."""
    os.makedirs(audio_dir, exist_ok=True)
    audio_files = []
    for seg in segments:
        if not seg.get('new_text', '').strip():
            continue
        out_path = os.path.join(audio_dir, f"seg_{seg['id']:04d}.mp3")
        try:
            await generate_tts_segment(seg['new_text'], out_path, voice)
            audio_files.append({'path': out_path, 'start': seg['start'], 'end': seg['end']})
        except Exception as e:
            print(f"   Warning: TTS fallo en segmento {seg['id']}: {e}")
    return audio_files

def get_duration(file_path):
    """Obtiene duracion de un archivo de audio/video."""
    cmd = ['ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_format', file_path]
    result = subprocess.run(cmd, capture_output=True, text=True)
    data = json.loads(result.stdout)
    return float(data['format']['duration'])

def compose_final_video(video_path, audio_files, output_path):
    """Compone el video final con el audio doblado."""
    video_duration = get_duration(video_path)
    
    if not audio_files:
        shutil.copy2(video_path, output_path)
        return output_path
    
    # Crear filtro complejo para posicionar cada audio
    inputs = ['-i', video_path]
    filter_parts = []
    
    for i, af in enumerate(audio_files):
        inputs.extend(['-i', af['path']])
        delay_ms = int(af['start'] * 1000)
        filter_parts.append(f"[{i+1}:a]adelay={delay_ms}|{delay_ms}[a{i}]")
    
    # Mezclar todos los audios
    mix_inputs = ''.join(f'[a{i}]' for i in range(len(audio_files)))
    filter_parts.append(f"{mix_inputs}amix=inputs={len(audio_files)}:duration=longest:dropout_transition=0[aout]")
    
    filter_complex = ';'.join(filter_parts)
    
    cmd = ['ffmpeg', '-y'] + inputs + [
        '-filter_complex', filter_complex,
        '-map', '0:v', '-map', '[aout]',
        '-c:v', 'copy', '-c:a', 'aac', '-b:a', '192k',
        '-shortest', output_path
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        # Fallback: solo reemplazar audio con el primer segmento
        print(f"   Warning: Mix complejo fallo, usando fallback simple")
        cmd_simple = ['ffmpeg', '-y', '-i', video_path, '-i', audio_files[0]['path'],
                      '-c:v', 'copy', '-c:a', 'aac', '-map', '0:v', '-map', '1:a',
                      '-shortest', output_path]
        subprocess.run(cmd_simple, capture_output=True, text=True, check=True)
    
    return output_path

print("✅ Pipeline de doblaje cargado")

In [ ]:
#@title 5. 🚀 CORRER - Doblar todos los videos pendientes
#@markdown Voz para el doblaje:
VOZ = "es-MX-JorgeNeural" #@param ["es-MX-JorgeNeural", "es-MX-DaliaNeural", "es-ES-AlvaroNeural", "es-AR-TomasNeural", "es-CO-GonzaloNeural"]

import time
from google.colab import files as colab_files

if not pending:
    print("⚠️ No hay videos pendientes en la Sheet")
    print("   Pega un link de YouTube en la columna A y vuelve a correr la celda 3")
else:
    for item in pending:
        row_num = item['row']
        url = item['url']
        
        print(f"\n{'='*60}")
        print(f"🎬 Procesando fila {row_num}: {url}")
        print(f"{'='*60}")
        
        # Actualizar estado en Sheet
        sheet.update_cell(row_num, 2, 'Procesando...')
        
        try:
            work_dir = os.path.join(TEMP_DIR, f'video_{row_num}')
            os.makedirs(work_dir, exist_ok=True)
            
            # Paso 1: Descargar
            print("📥 Descargando video...")
            start = time.time()
            video_path = download_video(url, work_dir)
            print(f"   Listo ({time.time()-start:.0f}s)")
            
            # Paso 2: Extraer audio
            print("🔊 Extrayendo audio...")
            audio_path = extract_audio(video_path)
            
            # Paso 3: Transcribir
            print("🎤 Transcribiendo con Whisper...")
            start = time.time()
            segments, lang = transcribe_audio(audio_path)
            print(f"   {len(segments)} segmentos, idioma: {lang} ({time.time()-start:.0f}s)")
            
            # Paso 4: Reescribir
            print("✍️ Preparando dialogos...")
            segments = rewrite_segments_spanish(segments)
            
            # Paso 5: Generar TTS
            print(f"🗣️ Generando voz ({VOZ})...")
            start = time.time()
            audio_dir = os.path.join(work_dir, 'tts')
            audio_files = await generate_all_tts(segments, audio_dir, VOZ)
            print(f"   {len(audio_files)} audios generados ({time.time()-start:.0f}s)")
            
            # Paso 6: Componer video
            print("🎬 Componiendo video final...")
            start = time.time()
            output_name = f"doblado_fila{row_num}_{int(time.time())}.mp4"
            output_path = os.path.join(work_dir, output_name)
            compose_final_video(video_path, audio_files, output_path)
            print(f"   Listo ({time.time()-start:.0f}s)")
            
            # Paso 7: Copiar a Drive
            print("☁️ Subiendo a Google Drive...")
            drive_path = os.path.join(DRIVE_OUTPUT, output_name)
            shutil.copy2(output_path, drive_path)
            
            # Actualizar Sheet
            sheet.update_cell(row_num, 2, 'Listo')
            sheet.update_cell(row_num, 3, f'Drive > VideosTraducidos > {output_name}')
            
            print(f"\n✅ Video doblado guardado en Drive!")
            print(f"   📁 Google Drive > VideosTraducidos > {output_name}")
            
            # Limpiar temp
            shutil.rmtree(work_dir, ignore_errors=True)
            
        except Exception as e:
            print(f"\n❌ Error: {e}")
            sheet.update_cell(row_num, 2, f'Error: {str(e)[:100]}')
    
    print(f"\n{'='*60}")
    print(f"🎉 Todos los videos procesados!")
    print(f"   Revisa tu Google Drive > VideosTraducidos")
    print(f"{'='*60}")